# 01 · The data and the acquisition simulator

What this notebook establishes, before any model is trained:

1. what the benchmark images look like once they have been through a simulated acquisition,
2. that lowering the dose degrades them the way photon statistics say it should,
3. what each of the five perturbation families actually does to an image.

The benchmark image is treated as **the object**, not as an image that already came out of a
scanner. Everything downstream is the output of one simulated acquisition.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # works without installing the package

CONFIG = "../configs/default.yaml"   # switch to ../configs/smoke.yaml to run offline in seconds


In [ ]:
from ctxaiqc.data import load_dataset
from ctxaiqc.utils import load_config

cfg = load_config(CONFIG)
data = load_dataset(**cfg["dataset"], seed=cfg["seed"])
data.summary()

## The object and its reference acquisition

Left: the raw benchmark image. Middle: one reference acquisition of it. Right: the difference.
Filtered back-projection is not the identity — it blurs, it rings, and it discards the corners
outside the reconstruction circle. That is exactly why the classifier is trained on the middle
image and not on the left one.

In [ ]:
from ctxaiqc.perturb import reference_acquisition

raw = data.x_test_raw[0, 0]
ref = reference_acquisition(raw, seed=0)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, img, title in zip(
    axes,
    [raw, ref, ref - raw],
    ["object (benchmark image)", "reference acquisition", "difference"],
):
    m = ax.imshow(img, cmap="gray" if "difference" not in title else "coolwarm")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    plt.colorbar(m, ax=ax, fraction=0.046)
fig.tight_layout()

## Dose reduction

`I0 exp(-p)` photons reach the detector, the count is Poisson-distributed, and the dose fraction
scales `I0`. Nothing else in the chain changes.

In [ ]:
from ctxaiqc.perturb import apply_perturbation, levels

doses = list(levels("dose"))
fig, axes = plt.subplots(1, len(doses), figsize=(2.2 * len(doses), 2.6))
for ax, d in zip(axes, doses):
    ax.imshow(apply_perturbation(raw, "dose", d, seed=1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{d:g} x dose", fontsize=9)
    ax.axis("off")
fig.tight_layout()

### Does the noise behave?

Photon noise in the reconstructed image should grow roughly as the inverse square root of the dose.
The simulator is not calibrated to any scanner, so what matters is the trend, not the constant.

In [ ]:
ref_noiseless = apply_perturbation(raw, "dose", 1.0, seed=0)
rmse = [
    float(np.sqrt(np.mean((apply_perturbation(raw, "dose", d, seed=k) - ref_noiseless) ** 2)))
    for d in doses for k in [7]
]

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.loglog(doses, rmse, "o-", label="measured")
ax.loglog(doses, rmse[0] * np.array(doses) ** -0.5 / doses[0] ** -0.5, "--", alpha=0.6,
          label=r"$\propto$ dose$^{-1/2}$")
ax.set_xlabel("dose fraction"); ax.set_ylabel("RMSE vs reference")
ax.invert_xaxis(); ax.legend(); ax.grid(alpha=0.3)

## The five families side by side

Each panel is the strongest level of one family. The reference acquisition is repeated first for
comparison.

In [ ]:
from ctxaiqc.perturb import PERTURBATIONS

fams = list(PERTURBATIONS)
fig, axes = plt.subplots(2, len(fams) + 1, figsize=(2.1 * (len(fams) + 1), 4.6))
axes[0, 0].imshow(ref, cmap="gray", vmin=0, vmax=1); axes[0, 0].set_title("reference", fontsize=8)
axes[1, 0].axis("off"); axes[0, 0].axis("off")
for j, fam in enumerate(fams, start=1):
    strongest = PERTURBATIONS[fam]["levels"][-1]
    img = apply_perturbation(raw, fam, strongest, seed=0)
    axes[0, j].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[0, j].set_title(f"{fam}\n{strongest}", fontsize=8)
    axes[1, j].imshow(img - ref, cmap="coolwarm", vmin=-0.2, vmax=0.2)
    axes[1, j].set_title("difference", fontsize=8)
    axes[0, j].axis("off"); axes[1, j].axis("off")
fig.tight_layout()

Every family reproduces the reference acquisition exactly at its first level, by construction:
that is the fairness condition the whole protocol rests on.

In [ ]:
from ctxaiqc.perturb import baseline_level

for fam in fams:
    base = apply_perturbation(raw, fam, baseline_level(fam), seed=0)
    print(f"{fam:>9}  RMSE(level 0 vs reference) = {np.sqrt(np.mean((base - ref) ** 2)):.2e}")